In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict

# ===================== CONFIG =====================
DATASET_ROOT = '/kaggle/input/cropped-plantdoc/PlantDoc_Cropped/train'
OUTPUT_ROOT = '/kaggle/working/fewshot_dataset'
SEED = 42
BASE_RATIO = 0.67  
NOVEL_RATIO = 0.22  
VAL_RATIO = 0.11   

K_SHOTS = [1, 5, 10, 20]  # Different shot settings
N_QUERY = 30  

random.seed(SEED)

# ===================== STEP 1: Analyze Dataset =====================
def analyze_dataset(dataset_path):
    """Đếm số ảnh mỗi class"""
    class_counts = {}
    
    for cls_name in os.listdir(dataset_path):
        cls_path = os.path.join(dataset_path, cls_name)
        if not os.path.isdir(cls_path):
            continue
        
        images = [f for f in os.listdir(cls_path) 
                 if f.endswith(('.jpg', '.png', '.jpeg'))]
        class_counts[cls_name] = len(images)
    
    return class_counts

print("="*70)
print("ANALYZING DATASET")
print("="*70)

class_counts = analyze_dataset(DATASET_ROOT)
all_classes = sorted(class_counts.keys())

print(f"\nTotal Classes: {len(all_classes)}")
print(f"Total Images: {sum(class_counts.values())}")
print(f"\nClass Distribution:")
for cls, count in sorted(class_counts.items(), key=lambda x: x[1]):
    print(f"  {cls:40s}: {count:4d} images")

# ===================== STEP 2: Stratified Class Split =====================
def stratified_class_split(classes, class_counts, base_r, novel_r, val_r):
    """
    Chia classes sao cho base/novel/val có phân bố số ảnh tương tự
    """
    # Sort classes by count
    sorted_classes = sorted(classes, key=lambda x: class_counts[x])
    
    n_total = len(sorted_classes)
    n_base = int(n_total * base_r)
    n_novel = int(n_total * novel_r)
    n_val = n_total - n_base - n_novel
    
    # Interleaved sampling (đảm bảo balanced distribution)
    base_classes = []
    novel_classes = []
    val_classes = []
    
    for i, cls in enumerate(sorted_classes):
        if i % 3 == 0 and len(base_classes) < n_base:
            base_classes.append(cls)
        elif i % 3 == 1 and len(novel_classes) < n_novel:
            novel_classes.append(cls)
        elif len(val_classes) < n_val:
            val_classes.append(cls)
        else:
            base_classes.append(cls)  # Remaining
    
    return base_classes, novel_classes, val_classes

print("\n" + "="*70)
print("SPLITTING CLASSES")
print("="*70)

base_classes, novel_classes, val_classes = stratified_class_split(
    all_classes, class_counts, BASE_RATIO, NOVEL_RATIO, VAL_RATIO
)

print(f"\n Base Classes ({len(base_classes)}):")
for cls in base_classes:
    print(f"   - {cls:40s} ({class_counts[cls]:3d} images)")

print(f"\n Novel Classes ({len(novel_classes)}):")
for cls in novel_classes:
    print(f"   - {cls:40s} ({class_counts[cls]:3d} images)")

print(f"\n Val Classes ({len(val_classes)}):")
for cls in val_classes:
    print(f"   - {cls:40s} ({class_counts[cls]:3d} images)")

# Statistics
base_imgs = sum(class_counts[c] for c in base_classes)
novel_imgs = sum(class_counts[c] for c in novel_classes)
val_imgs = sum(class_counts[c] for c in val_classes)

print(f"\n Statistics:")
print(f"   Base:  {len(base_classes):2d} classes, {base_imgs:4d} images")
print(f"   Novel: {len(novel_classes):2d} classes, {novel_imgs:4d} images")
print(f"   Val:   {len(val_classes):2d} classes, {val_imgs:4d} images")

# ===================== STEP 3: Create Directory Structure =====================
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/base", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/novel", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/val", exist_ok=True)

print("\n" + "="*70)
print("📁 CREATING DIRECTORY STRUCTURE")
print("="*70)

# Copy base classes (toàn bộ images)
for cls in base_classes:
    src = os.path.join(DATASET_ROOT, cls)
    dst = os.path.join(OUTPUT_ROOT, 'base', cls)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Copied {cls} to base/")

# Copy val classes (toàn bộ images)
for cls in val_classes:
    src = os.path.join(DATASET_ROOT, cls)
    dst = os.path.join(OUTPUT_ROOT, 'val', cls)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"✅ Copied {cls} to val/")

# ===================== STEP 4: Prepare Novel Classes Support/Query =====================
print("\n" + "="*70)
print(" PREPARING NOVEL CLASSES (Support/Query Split)")
print("="*70)

novel_splits = {}

for cls in novel_classes:
    cls_path = os.path.join(DATASET_ROOT, cls)
    images = [f for f in os.listdir(cls_path) 
             if f.endswith(('.jpg', '.png', '.jpeg'))]
    random.shuffle(images)
    
    # Reserve query set first (30 images)
    query_images = images[:N_QUERY]
    support_pool = images[N_QUERY:]
    
    # Create support sets for different K-shots
    novel_splits[cls] = {
        'query': query_images,
        'support_pool': support_pool
    }
    
    # Create directories
    cls_dir = os.path.join(OUTPUT_ROOT, 'novel', cls)
    os.makedirs(f"{cls_dir}/query", exist_ok=True)
    
    # Copy query images
    for img in query_images:
        src = os.path.join(cls_path, img)
        dst = os.path.join(cls_dir, 'query', img)
        shutil.copy2(src, dst)
    
    # Create K-shot support sets
    for k in K_SHOTS:
        if len(support_pool) >= k:
            os.makedirs(f"{cls_dir}/{k}shot", exist_ok=True)
            shot_images = support_pool[:k]
            
            for img in shot_images:
                src = os.path.join(cls_path, img)
                dst = os.path.join(cls_dir, f'{k}shot', img)
                shutil.copy2(src, dst)
            
            novel_splits[cls][f'{k}shot'] = shot_images
    
    print(f" {cls}:")
    print(f"   - Query: {len(query_images)} images")
    for k in K_SHOTS:
        if f'{k}shot' in novel_splits[cls]:
            print(f"   - {k}-shot: {len(novel_splits[cls][f'{k}shot'])} images")


print("\n" + "="*70)
print(" SAVING METADATA")
print("="*70)

metadata = {
    'total_classes': len(all_classes),
    'base_classes': base_classes,
    'novel_classes': novel_classes,
    'val_classes': val_classes,
    'class_counts': class_counts,
    'novel_splits': novel_splits,
    'config': {
        'k_shots': K_SHOTS,
        'n_query': N_QUERY,
        'seed': SEED
    }
}

with open(f'{OUTPUT_ROOT}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f" Saved metadata to {OUTPUT_ROOT}/metadata.json")

# Create simple text files for easy reference
with open(f'{OUTPUT_ROOT}/base_classes.txt', 'w') as f:
    f.write('\n'.join(base_classes))

with open(f'{OUTPUT_ROOT}/novel_classes.txt', 'w') as f:
    f.write('\n'.join(novel_classes))

with open(f'{OUTPUT_ROOT}/val_classes.txt', 'w') as f:
    f.write('\n'.join(val_classes))

print(f" Saved class lists to text files")


print("\n Ready for few-shot learning!")
print("="*70)

📊 ANALYZING DATASET

Total Classes: 27
Total Images: 8443

Class Distribution:
  Corn_Gray_leaf_spot                     :   75 images
  Corn_rust_leaf                          :  117 images
  grape_leaf_black_rot                    :  125 images
  Apple_Scab_Leaf                         :  158 images
  Apple_rust_leaf                         :  168 images
  Tomato_Early_blight_leaf                :  195 images
  grape_leaf                              :  205 images
  Tomato_leaf_late_blight                 :  207 images
  Cherry_leaf                             :  220 images
  Tomato_leaf_mosaic_virus                :  225 images
  Apple_leaf                              :  237 images
  Potato_leaf_late_blight                 :  240 images
  Soyabean_leaf                           :  246 images
  Squash_Powdery_mildew_leaf              :  248 images
  Bell_pepper_leaf_spot                   :  248 images
  Tomato_leaf_bacterial_spot              :  266 images
  Tomato_mold_leaf       

In [ ]:
"""
FEW-SHOT SUPCON PLANT DISEASE CLASSIFICATION SUA FOCAL LOSS
Minimal modifications to existing code for few-shot learning
"""
import os, random, time, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedShuffleSplit
import seaborn as sns
import json

# ===================== CONFIG =====================
DATASET_ROOT_BASE = '/kaggle/working/fewshot_dataset/base'  
DATASET_ROOT_NOVEL = '/kaggle/working/fewshot_dataset/novel'  
DATASET_ROOT_VAL = '/kaggle/working/fewshot_dataset/val'  
DATASET_ROOT_TEST = '/kaggle/input/cropped-plantdoc/PlantDoc_Cropped/test'

IMG_SIZE = 224
TEMPERATURE = 0.1
PROJ_DIM = 128
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
VAL_SPLIT = 0.2  # Val split TRONG base classes
NUM_WORKERS = 4
PATIENCE = 6

BASE_BATCH = 128
BASE_EPOCHS_PRETRAIN = 40
BASE_EPOCHS_LINEAR = 40
LR_PRETRAIN = 3e-4
LR_LINEAR = 1e-4
WEIGHT_DECAY = 5e-5

USE_AMP = True
FINETUNE_BACKBONE = False
DROPOUT_P = 0.5
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA = 0.1
MIXUP_PROB = 0.3
FOCAL_LOSS_GAMMA = 2.0
TTA_AUGMENTS = 3
USE_FOCAL_LOSS = True
HN_THRESHOLD = 0.85

#  FEW-SHOT CONFIGS
FEWSHOT_EVAL = True  # Enable few-shot evaluation
FEWSHOT_N_WAY = 5  # N-way classification
FEWSHOT_K_SHOTS = [1, 5, 10]  # Different shot settings
FEWSHOT_EPISODES = 600  # Episodes for evaluation

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==================== Transforms (SAME AS BEFORE) ====================
train_augment = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=25),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.25),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225))
])

class TwoCropTransform:
    def __init__(self, base_transform): 
        self.base_transform = base_transform
    def __call__(self, x): 
        return [self.base_transform(x), self.base_transform(x)]


class SoftMixup:
    def __init__(self, alpha=0.1, prob=0.3):
        self.alpha = alpha
        self.prob = prob
    
    def __call__(self, images, labels):
        if self.alpha <= 0 or np.random.rand() > self.prob:
            return images, labels, None
        
        lam = np.random.beta(self.alpha, self.alpha)
        lam = max(lam, 1 - lam)
        
        batch_size = images.size(0)
        index = torch.randperm(batch_size).to(images.device)
        
        mixed_images = lam * images + (1 - lam) * images[index]
        labels_a = labels
        labels_b = labels[index]
        
        return mixed_images, (labels_a, labels_b, lam), index

# ==================== Focal Loss (SAME) ====================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


def stratified_split_dataset(dataset_path, val_split=0.2, seed=42):
    full_dataset = datasets.ImageFolder(dataset_path)
    labels = [label for _, label in full_dataset.imgs]
    
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_split, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    
    print(f" Stratified split: train={len(train_idx)}, val={len(val_idx)}")
    return train_idx, val_idx, full_dataset.classes

# ==================== Dataset Loading ====================
#  THAY ĐỔI: Load BASE classes thay vì toàn bộ
full_train_plain = datasets.ImageFolder(DATASET_ROOT_BASE)
num_samples = len(full_train_plain)

if num_samples < 5000:
    BATCH_SIZE = min(32, BASE_BATCH)
    EPOCHS_PRETRAIN = min(25, BASE_EPOCHS_PRETRAIN)
    EPOCHS_LINEAR = min(25, BASE_EPOCHS_LINEAR)
else:
    BATCH_SIZE = BASE_BATCH
    EPOCHS_PRETRAIN = BASE_EPOCHS_PRETRAIN
    EPOCHS_LINEAR = BASE_EPOCHS_LINEAR

# Stratified split
train_idx, val_idx, base_class_names = stratified_split_dataset(
    DATASET_ROOT_BASE, VAL_SPLIT, SEED
)

print("\n" + "="*70)
print(" BASE CLASS NAMES (for meta-training)")
print("="*70)
for i, name in enumerate(base_class_names):
    print(f"Index {i:2d}: {name}")
print("="*70)

# Create datasets
train_dataset = datasets.ImageFolder(DATASET_ROOT_BASE, transform=TwoCropTransform(train_augment))
train_linear_full = datasets.ImageFolder(DATASET_ROOT_BASE, transform=eval_transform)

# Create subsets
train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(train_dataset, val_idx)
train_linear_subset = Subset(train_linear_full, train_idx)
val_linear_subset = Subset(train_linear_full, val_idx)

# Weighted Sampler
labels_for_sampler = np.array([full_train_plain.targets[i] for i in train_idx])
class_sample_count = np.bincount(labels_for_sampler, minlength=len(base_class_names))
class_sample_count = np.where(class_sample_count == 0, 1, class_sample_count)
weights = 1.0 / class_sample_count
samples_weight = weights[labels_for_sampler]
samples_weight = torch.from_numpy(samples_weight).double()
sampler_pre = WeightedRandomSampler(samples_weight, num_samples=len(samples_weight), replacement=True)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler_pre, 
                         num_workers=NUM_WORKERS, drop_last=True)
val_loader_for_loss = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, 
                                num_workers=NUM_WORKERS)
train_linear_loader = DataLoader(train_linear_subset, batch_size=BATCH_SIZE, shuffle=True, 
                                num_workers=NUM_WORKERS)
val_linear_loader = DataLoader(val_linear_subset, batch_size=BATCH_SIZE, shuffle=False, 
                              num_workers=NUM_WORKERS)

print(f"\n Dataset sizes:")
print(f"   Base train: {len(train_subset)}")
print(f"   Base val:   {len(val_subset)}")

# ==================== Model (SAME AS BEFORE) ====================
base_net = models.efficientnet_b0(pretrained=True)

class SupConNet(nn.Module):
    def __init__(self, backbone, feat_dim, proj_dim=128, num_classes=None):
        super().__init__()
        self.backbone = backbone
        
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, proj_dim)
        )
        
        self.classifier = None
        if num_classes is not None:
            self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x, return_logits=False):
        feats = self.backbone(x)
        z = F.normalize(self.proj(feats), dim=1)

        if return_logits and self.classifier is not None:
            logits = self.classifier(feats)
            return z, logits
        
        return z


NUM_BASE_CLASSES = len(base_class_names)

model = SupConNet(
    feature_extractor_msfe,
    feat_dim,
    PROJ_DIM,
    num_classes=NUM_BASE_CLASSES
).to(DEVICE)

focal_criterion = FocalLoss(
    gamma=FOCAL_LOSS_GAMMA,
    reduction='mean'
)


# ==================== SupCon Loss ====================
class SupConLoss(nn.Module):
    def __init__(self, temperature=TEMPERATURE, hnm_threshold=HN_THRESHOLD):
        super().__init__()
        self.temperature = temperature
        self.hnm_threshold = hnm_threshold
    
    def forward(self, features, labels=None):
        bsz, n_views, dim = features.shape
        device = features.device
        features = F.normalize(features, dim=2)
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        
        raw_sim = torch.matmul(contrast_feature, contrast_feature.T)
        logits = raw_sim / self.temperature
        
        if labels is None:
             mask = torch.eye(bsz, device=device)
        else:
             labels = labels.contiguous().view(-1,1)
             mask = torch.eq(labels, labels.T).float().to(device)
        
        mask = mask.repeat(n_views, n_views)
        logits_mask = (torch.ones_like(mask) - torch.eye(mask.shape[0], device=device))
        
        neg_mask = 1 - mask
        easy_negatives = (raw_sim < self.hnm_threshold) & (neg_mask.bool())
        
        hnm_logits = logits.clone()
        HN_REPLACEMENT_VALUE = -1e4
        hnm_logits[easy_negatives] = HN_REPLACEMENT_VALUE
        
        logits_max, _ = torch.max(hnm_logits, dim=1, keepdim=True)
        hnm_logits -= logits_max.detach()
        
        exp_logits = torch.exp(hnm_logits) * logits_mask
        log_prob = hnm_logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)
        
        mask *= logits_mask
        mask_pos_pairs = torch.where(mask.sum(1) < 1e-6, torch.ones_like(mask.sum(1)), mask.sum(1))
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs
        loss = - mean_log_prob_pos.view(n_views, bsz).mean()
        
        return loss

criterion = SupConLoss(temperature=TEMPERATURE, hnm_threshold=HN_THRESHOLD)
optimizer = optim.AdamW(model.parameters(), lr=LR_PRETRAIN, weight_decay=WEIGHT_DECAY)
scheduler_pre = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_PRETRAIN, eta_min=1e-6
)

# ==================== Pretrain Loop (SAME) ====================
print('\n' + '='*70)
print(' Stage 1:  Pretraining on BASE classes')
print('='*70)

scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
history = {'pretrain_train_loss': [], 'pretrain_val_loss': []}
best_val_loss = float('inf')
best_encoder_wts = copy.deepcopy(model.state_dict())
pretrain_counter = 0

for epoch in range(1, EPOCHS_PRETRAIN+1):
    model.train()
    run_loss = 0
    
    for (images, labels) in train_loader:
        im1, im2 = images[0].to(DEVICE), images[1].to(DEVICE)
        labels = labels.to(DEVICE)
        b = labels.size(0)
        inputs = torch.cat([im1, im2], dim=0)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            z, logits = model(inputs, return_logits=True)
            
            f1, f2 = torch.split(z, [b, b])
            supcon_loss = criterion(
                torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1),
                labels
            )
        
            #  Focal loss (chỉ dùng view đầu)
            cls_loss = focal_criterion(logits[:b], labels)
        
            loss = supcon_loss + 0.5 * cls_loss   # λ = 0.5 (khuyến nghị)

        
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item()
    
    scheduler_pre.step()
    train_loss = run_loss / max(1, len(train_loader))
    history['pretrain_train_loss'].append(train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for (vimg, vlab) in val_loader_for_loss:
            v1, v2 = vimg[0].to(DEVICE), vimg[1].to(DEVICE)
            vlab = vlab.to(DEVICE)
            b = vlab.size(0)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                z, logits = model(torch.cat([v1, v2], dim=0), return_logits=True)
                
                f1, f2 = torch.split(z, [b, b])
                supcon_loss = criterion(
                    torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1),
                    vlab
                )
                
                cls_loss = focal_criterion(logits[:b], vlab)
                loss = supcon_loss + 0.5 * cls_loss

            val_loss += loss.item()
    
    val_loss /= max(1,len(val_loader_for_loss))
    history['pretrain_val_loss'].append(val_loss)
    
    print(f"[Pretrain] Epoch {epoch:2d}/{EPOCHS_PRETRAIN} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        best_encoder_wts = copy.deepcopy(model.state_dict())
        pretrain_counter = 0
        torch.save(best_encoder_wts, '/kaggle/working/fewshot_encoder_best.pth')
    else:
        pretrain_counter += 1
        if pretrain_counter >= PATIENCE:
            print(f'✋ Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_encoder_wts)

# ====================  NEW: FEW-SHOT EVALUATION FUNCTIONS ====================
def compute_prototypes(support_features, support_labels):
    """Tính prototype bằng cách normalize và lấy trung bình"""
    prototypes = {}
    unique_labels = torch.unique(support_labels)
    
    for label in unique_labels:
        mask = (support_labels == label)
        class_features = support_features[mask]
        
        # Normalize từng feature trước khi trung bình cộng
        class_features = F.normalize(class_features, dim=1)
        prototype = class_features.mean(dim=0)
        
        # Normalize lại prototype một lần nữa để nằm trên mặt cầu
        prototypes[label.item()] = F.normalize(prototype, dim=0)
    
    return prototypes

def classify_by_prototypes(query_features, prototypes):
    """Phân loại dựa trên Cosine Similarity (Dot Product)"""
    # Normalize query features
    query_features = F.normalize(query_features, dim=1)
    
    # Chuyển dictionary prototypes thành tensor để tính toán song song
    proto_labels = list(prototypes.keys())
    proto_tensor = torch.stack([prototypes[l] for l in proto_labels]) # [N_way, Dim]
    
    # Tính tương đồng: [N_query, Dim] x [Dim, N_way] = [N_query, N_way]
    similarities = torch.matmul(query_features, proto_tensor.T)
    
    # Chọn lớp có độ tương đồng cao nhất
    pred_indices = torch.argmax(similarities, dim=1)
    predictions = torch.tensor([proto_labels[i] for i in pred_indices])
    
    return predictions

def sample_fewshot_episode(dataset_path, n_way, k_shot, seed=None):

    if seed is not None:
        random.seed(seed)
    
    # Load dataset
    dataset = datasets.ImageFolder(dataset_path, transform=eval_transform)
    classes = dataset.classes
    
    # Sample N classes
    sampled_classes = random.sample(range(len(classes)), n_way)
    
    support_images = []
    support_labels = []
    query_images = []
    query_labels = []
    
    for new_label, original_label in enumerate(sampled_classes):
        # Get all indices for this class
        class_indices = [i for i, (_, label) in enumerate(dataset.samples) 
                        if label == original_label]
        
        # Sample K+Q images
        n_query = min(15, len(class_indices) - k_shot)
        sampled_indices = random.sample(class_indices, k_shot + n_query)
        
        support_indices = sampled_indices[:k_shot]
        query_indices = sampled_indices[k_shot:k_shot + n_query]
        
        # Load images
        for idx in support_indices:
            img, _ = dataset[idx]
            support_images.append(img)
            support_labels.append(new_label)
        
        for idx in query_indices:
            img, _ = dataset[idx]
            query_images.append(img)
            query_labels.append(new_label)
    
    support_images = torch.stack(support_images)
    support_labels = torch.tensor(support_labels)
    query_images = torch.stack(query_images)
    query_labels = torch.tensor(query_labels)
    
    return support_images, support_labels, query_images, query_labels


# ====================  NEW: FEW-SHOT EVALUATION ====================
def evaluate_fewshot(encoder, dataset_path, n_way, k_shot, num_episodes):
    encoder.eval()
    device = next(encoder.parameters()).device
    
    # 1. Trích xuất toàn bộ đặc trưng một lần duy nhất
    print(" Pre-extracting features to avoid CPU bottleneck...")
    all_features = []
    all_labels = []
    dataset = datasets.ImageFolder(dataset_path, transform=eval_transform)
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    
    with torch.no_grad():
        for imgs, labs in loader:
            feats = encoder(imgs.to(device))
            all_features.append(feats.cpu()) # Lưu vào RAM để tiết kiệm VRAM
            all_labels.append(labs)
            
    all_features = torch.cat(all_features)
    all_labels = torch.cat(all_labels)
    unique_labels = torch.unique(all_labels)

    # 2. Chạy episode trên các vector đã có sẵn (không đọc ổ đĩa nữa)
    episode_accs = []
    for ep in range(num_episodes):
        # Chọn ngẫu nhiên N lớp
        sampled_classes = np.random.choice(unique_labels, n_way, replace=False)
        
        s_feats_list, q_feats_list = [], []
        s_labs_list, q_labs_list = [], []

        for i, cls in enumerate(sampled_classes):
            cls_indices = (all_labels == cls).nonzero(as_tuple=True)[0]
            # Lấy ngẫu nhiên k_shot + n_query ảnh
            perm = torch.randperm(len(cls_indices))
            s_idx = cls_indices[perm[:k_shot]]
            q_idx = cls_indices[perm[k_shot:k_shot+15]] # Mặc định 15 query
            
            s_feats_list.append(all_features[s_idx])
            q_feats_list.append(all_features[q_idx])
            s_labs_list.append(torch.full((len(s_idx),), i))
            q_labs_list.append(torch.full((len(q_idx),), i))

        # Tính toán nhanh trên GPU/RAM
        s_f = torch.cat(s_feats_list).to(device)
        q_f = torch.cat(q_feats_list).to(device)
        s_l = torch.cat(s_labs_list).to(device)
        q_l = torch.cat(q_labs_list).to(device)

        prototypes = compute_prototypes(s_f, s_l)
        preds = classify_by_prototypes(q_f, prototypes).to(device)
        acc = (preds == q_l).sum().item() / len(q_l)
        episode_accs.append(acc)

    return np.mean(episode_accs), np.std(episode_accs), 1.96*np.std(episode_accs)/np.sqrt(num_episodes), episode_accs

if FEWSHOT_EVAL:

    print('\n' + '='*70)
    print('Stage 2: Few-Shot Evaluation on NOVEL classes')
    print('='*70)

    model.load_state_dict(best_encoder_wts)
    encoder = model.backbone
    encoder.eval()

    fewshot_results = {}

    for k in FEWSHOT_K_SHOTS:
        mean, std, ci95, accs = evaluate_fewshot(
            encoder,
            DATASET_ROOT_NOVEL,
            FEWSHOT_N_WAY,
            k,
            FEWSHOT_EPISODES
        )

        fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot'] = {
            'mean': mean,
            'std': std,
            'ci95': ci95
        }

        print(f"\n{FEWSHOT_N_WAY}-way {k}-shot:")
        print(f"Accuracy: {mean:.4f} ± {ci95:.4f}")

    
    # Save results
    with open('/kaggle/working/fewshot_results.json', 'w') as f:
        json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'accuracies'} 
                  for k, v in fewshot_results.items()}, f, indent=2)
    
    # Plot results
    fig, ax = plt.subplots(figsize=(10, 6))
    
    k_shots = FEWSHOT_K_SHOTS
    means = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['mean'] for k in k_shots]
    stds = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['std'] for k in k_shots]
    ci95s = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['ci95'] for k in k_shots]
    
    ax.plot(k_shots, means, 'o-', linewidth=2, markersize=10, color='#2ecc71', label='Mean Accuracy')
    ax.fill_between(k_shots, 
                    [m - c for m, c in zip(means, ci95s)],
                    [m + c for m, c in zip(means, ci95s)],
                    alpha=0.3, color='#2ecc71', label='95% CI')
    
    ax.set_xlabel('K-shot', fontsize=14, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
    ax.set_title(f'{FEWSHOT_N_WAY}-Way Few-Shot Performance', fontsize=16, fontweight='bold')
    ax.set_xticks(k_shots)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    
    # Add value labels
    for k, m in zip(k_shots, means):
        ax.text(k, m + 0.03, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/fewshot_performance.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n Saved few-shot results to:")
    print(f"   - /kaggle/working/fewshot_results.json")
    print(f"   - /kaggle/working/fewshot_performance.png")

# ==================== Final Summary ====================
print("\n" + "="*70)
print(" FEW-SHOT TRAINING COMPLETE!")
print("="*70)

if FEWSHOT_EVAL:
    print("\n FEW-SHOT RESULTS:")
    print("="*70)
    for k_shot in FEWSHOT_K_SHOTS:
        result = fewshot_results[f'{FEWSHOT_N_WAY}way_{k_shot}shot']
        print(f"{FEWSHOT_N_WAY}-way {k_shot}-shot: {result['mean']:.4f} ± {result['ci95']:.4f}")
    print("="*70)

🌿 FEW-SHOT SUPCON PLANT DISEASE CLASSIFICATION
Meta-Baseline + Prototypical Networks
✅ Stratified split: train=5348, val=1338

🎯 BASE CLASS NAMES (for meta-training)
Index  0: Apple_Scab_Leaf
Index  1: Bell_pepper_leaf
Index  2: Blueberry_leaf
Index  3: Corn_Gray_leaf_spot
Index  4: Corn_leaf_blight
Index  5: Peach_leaf
Index  6: Potato_leaf_early_blight
Index  7: Raspberry_leaf
Index  8: Soyabean_leaf
Index  9: Squash_Powdery_mildew_leaf
Index 10: Strawberry_leaf
Index 11: Tomato_Septoria_leaf_spot
Index 12: Tomato_leaf
Index 13: Tomato_leaf_bacterial_spot
Index 14: Tomato_leaf_mosaic_virus
Index 15: Tomato_leaf_yellow_virus
Index 16: Tomato_mold_leaf
Index 17: grape_leaf

📊 Dataset sizes:
   Base train: 5348
   Base val:   1338

🚀 Stage 1:  Pretraining on BASE classes


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[Pretrain] Epoch  1/40 | Train: 3.7727 | Val: 3.5935
[Pretrain] Epoch  2/40 | Train: 3.3199 | Val: 3.4462
[Pretrain] Epoch  3/40 | Train: 3.1279 | Val: 3.3228
[Pretrain] Epoch  4/40 | Train: 3.0307 | Val: 3.3166
[Pretrain] Epoch  5/40 | Train: 2.9442 | Val: 3.2563
[Pretrain] Epoch  6/40 | Train: 2.8659 | Val: 3.2008
[Pretrain] Epoch  7/40 | Train: 2.8152 | Val: 3.1916
[Pretrain] Epoch  8/40 | Train: 2.7765 | Val: 3.1888
[Pretrain] Epoch  9/40 | Train: 2.7663 | Val: 3.1688
[Pretrain] Epoch 10/40 | Train: 2.7499 | Val: 3.1935
[Pretrain] Epoch 11/40 | Train: 2.7286 | Val: 3.1746
[Pretrain] Epoch 12/40 | Train: 2.7132 | Val: 3.1408
[Pretrain] Epoch 13/40 | Train: 2.7129 | Val: 3.1457
[Pretrain] Epoch 14/40 | Train: 2.7137 | Val: 3.1532
[Pretrain] Epoch 15/40 | Train: 2.6981 | Val: 3.1397
[Pretrain] Epoch 16/40 | Train: 2.6976 | Val: 3.1287
[Pretrain] Epoch 17/40 | Train: 2.6829 | Val: 3.1348
[Pretrain] Epoch 18/40 | Train: 2.6805 | Val: 3.1147
[Pretrain] Epoch 19/40 | Train: 2.6837 | Val: 

KeyboardInterrupt: 

In [ ]:
def evaluate_fewshot(encoder, dataset_path, n_way, k_shot, num_episodes):
    encoder.eval()
    device = next(encoder.parameters()).device
    
    # 1. Trích xuất toàn bộ đặc trưng một lần duy nhất
    print(" Pre-extracting features to avoid CPU bottleneck...")
    all_features = []
    all_labels = []
    dataset = datasets.ImageFolder(dataset_path, transform=eval_transform)
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    
    with torch.no_grad():
        for imgs, labs in loader:
            feats = encoder(imgs.to(device))
            all_features.append(feats.cpu()) # Lưu vào RAM để tiết kiệm VRAM
            all_labels.append(labs)
            
    all_features = torch.cat(all_features)
    all_labels = torch.cat(all_labels)
    unique_labels = torch.unique(all_labels)

    # 2. Chạy episode trên các vector đã có sẵn (không đọc ổ đĩa nữa)
    episode_accs = []
    for ep in range(num_episodes):
        # Chọn ngẫu nhiên N lớp
        sampled_classes = np.random.choice(unique_labels, n_way, replace=False)
        
        s_feats_list, q_feats_list = [], []
        s_labs_list, q_labs_list = [], []

        for i, cls in enumerate(sampled_classes):
            cls_indices = (all_labels == cls).nonzero(as_tuple=True)[0]
            # Lấy ngẫu nhiên k_shot + n_query ảnh
            perm = torch.randperm(len(cls_indices))
            s_idx = cls_indices[perm[:k_shot]]
            q_idx = cls_indices[perm[k_shot:k_shot+15]] # Mặc định 15 query
            
            s_feats_list.append(all_features[s_idx])
            q_feats_list.append(all_features[q_idx])
            s_labs_list.append(torch.full((len(s_idx),), i))
            q_labs_list.append(torch.full((len(q_idx),), i))

        # Tính toán nhanh trên GPU/RAM
        s_f = torch.cat(s_feats_list).to(device)
        q_f = torch.cat(q_feats_list).to(device)
        s_l = torch.cat(s_labs_list).to(device)
        q_l = torch.cat(q_labs_list).to(device)

        prototypes = compute_prototypes(s_f, s_l)
        preds = classify_by_prototypes(q_f, prototypes).to(device)
        acc = (preds == q_l).sum().item() / len(q_l)
        episode_accs.append(acc)

    return np.mean(episode_accs), np.std(episode_accs), 1.96*np.std(episode_accs)/np.sqrt(num_episodes), episode_accs
if FEWSHOT_EVAL:

    print('\n' + '='*70)
    print('Stage 2: Few-Shot Evaluation on NOVEL classes')
    print('='*70)

    model.load_state_dict(best_encoder_wts)
    encoder = model.backbone
    encoder.eval()

    fewshot_results = {}

    for k in FEWSHOT_K_SHOTS:
        mean, std, ci95, accs = evaluate_fewshot(
            encoder,
            DATASET_ROOT_NOVEL,
            FEWSHOT_N_WAY,
            k,
            FEWSHOT_EPISODES
        )

        fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot'] = {
            'mean': mean,
            'std': std,
            'ci95': ci95
        }

        print(f"\n{FEWSHOT_N_WAY}-way {k}-shot:")
        print(f"Accuracy: {mean:.4f} ± {ci95:.4f}")

    
    # Save results
    with open('/kaggle/working/fewshot_results.json', 'w') as f:
        json.dump({k: {kk: vv for kk, vv in v.items() if kk != 'accuracies'} 
                  for k, v in fewshot_results.items()}, f, indent=2)
    
    # Plot results
    fig, ax = plt.subplots(figsize=(10, 6))
    
    k_shots = FEWSHOT_K_SHOTS
    means = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['mean'] for k in k_shots]
    stds = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['std'] for k in k_shots]
    ci95s = [fewshot_results[f'{FEWSHOT_N_WAY}way_{k}shot']['ci95'] for k in k_shots]
    
    ax.plot(k_shots, means, 'o-', linewidth=2, markersize=10, color='#2ecc71', label='Mean Accuracy')
    ax.fill_between(k_shots, 
                    [m - c for m, c in zip(means, ci95s)],
                    [m + c for m, c in zip(means, ci95s)],
                    alpha=0.3, color='#2ecc71', label='95% CI')
    
    ax.set_xlabel('K-shot', fontsize=14, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=14, fontweight='bold')
    ax.set_title(f'{FEWSHOT_N_WAY}-Way Few-Shot Performance', fontsize=16, fontweight='bold')
    ax.set_xticks(k_shots)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=12)
    
    # Add value labels
    for k, m in zip(k_shots, means):
        ax.text(k, m + 0.03, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/fewshot_performance.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"\n Saved few-shot results to:")
    print(f"   - /kaggle/working/fewshot_results.json")
    print(f"   - /kaggle/working/fewshot_performance.png")

# ==================== Final Summary ====================
print("\n" + "="*70)
print("🎉 FEW-SHOT TRAINING COMPLETE!")
print("="*70)

if FEWSHOT_EVAL:
    print("\n FEW-SHOT RESULTS:")
    print("="*70)
    for k_shot in FEWSHOT_K_SHOTS:
        result = fewshot_results[f'{FEWSHOT_N_WAY}way_{k_shot}shot']
        print(f"{FEWSHOT_N_WAY}-way {k_shot}-shot: {result['mean']:.4f} ± {result['ci95']:.4f}")
    print("="*70)


Stage 2: Few-Shot Evaluation on NOVEL classes
🚀 Pre-extracting features to avoid CPU bottleneck...

5-way 1-shot:
Accuracy: 0.6627 ± 0.0062
🚀 Pre-extracting features to avoid CPU bottleneck...

5-way 5-shot:
Accuracy: 0.8184 ± 0.0039
🚀 Pre-extracting features to avoid CPU bottleneck...

5-way 10-shot:
Accuracy: 0.8585 ± 0.0030

✅ Saved few-shot results to:
   - /kaggle/working/fewshot_results.json
   - /kaggle/working/fewshot_performance.png

🎉 FEW-SHOT TRAINING COMPLETE!

📊 FEW-SHOT RESULTS:
5-way 1-shot: 0.6627 ± 0.0062
5-way 5-shot: 0.8184 ± 0.0039
5-way 10-shot: 0.8585 ± 0.0030

✅ Key Achievements:
   - Trained encoder on 18 base classes
   - Evaluated on novel classes with 1/5/10-shot
   - No retraining needed for new diseases!

💡 Deployment:
   1. Load encoder: encoder = torch.load('fewshot_encoder_best.pth')
   2. Get 5 samples of new disease
   3. Compute prototype
   4. Classify immediately!
   
📁 Generated Files:
   - fewshot_encoder_best.pth
   - fewshot_results.json
   - f